In [ ]:
print("Rezvan")

# تشخیص پلاک خودرو — نسخه‌ی ۲ (صحنه‌ی شلوغ / بزرگراه)

نسخه‌ی سنگین‌ترِ `pelak.ipynb` برای ویدیوهایی که **وسایل نقلیه‌ی زیاد** دارند و مدل nano نصفشان را از دست می‌داد.

تفاوت‌ها با نسخه‌ی ۱:
1. مدل خودرو **`yolo11s`** به‌جای `yolo26n` — recall حدود ۳× بیشتر روی صحنه‌ی شلوغ
2. **`imgsz=1280`** — خودروهای دور/کوچک هم دیده می‌شوند (روی اجسام بزرگ هم بی‌مشکل)
3. **`VEHICLE_CONF=0.30`** — خودروهای با اطمینان متوسط هم برمی‌گردند
4. ردیابی با **ByteTrack** (`model.track`) — هر خودرو یک **ID پایدار** می‌گیرد، مقاوم به انسداد
5. اولویت‌بندیِ نمایش: **۱) خودرو همیشه تشخیص داده می‌شود ۲) جعبه‌ی پلاک همیشه نشان داده می‌شود اگر پیدا شود
   ۳) متن فقط وقتی نشان داده می‌شود که واقعاً درست خوانده شده باشد.** OCR سقف دائمی ندارد — تا وقتی خودرو
   دیده می‌شود و متن خوب قفل نشده، هر `PLATE_EVERY` فریم دوباره امتحان می‌شود (نسخه‌ی اولِ این ایده بعد از چند
   تلاشِ ناموفق برای همیشه از خودرو صرف‌نظر می‌کرد؛ رفع شد).

خط لوله:
- YOLO خودرو (`.track`) → داخل هر خودرو YOLO پلاک → OCR (Hezar فارسی / fast-plate لاتین)
- تحلیل کشوری بر اساس `COUNTRY_MODE` (`IR` / `GLOBAL` / `AUTO`)

ساختار: بخش ۱ نصب · ۲ تنظیمات · ۳ مدل‌ها · ۴ توابع کمکی · ۵ منطق پلاک ایران · ۶ پردازش ویدیو (ByteTrack) · ۷ تست · ۸ GIF

## بخش ۱ — نصب و ایمپورت

In [ ]:
# اگر از کرنل «Python (pelak)» استفاده می‌کنی این سلول لازم نیست (کامنت بماند).
# %pip install -q opencv-python ultralytics easyocr numpy torch torchvision
# %pip install -q hezar arabic-reshaper python-bidi      # OCR فارسی + رندر متن RTL روی ویدیو
# %pip install -q fast-plate-ocr onnxruntime             # OCR دقیق پلاک لاتین (۶۵+ کشور)
# %pip install -q lapx                                    # لازم برای ردیاب ByteTrack

In [ ]:
import csv
import urllib.request
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import torch
from ultralytics import YOLO
import easyocr

## بخش ۲ — تنظیمات

همه‌ی پارامترها و مسیرها فقط همین‌جا.

In [ ]:
# --- مسیرها (مطلق) ---
PROJECT_DIR = Path(r"C:\1\1_پروژه\تشخیص پلاکو ماشین")
VIDEO_DIR = PROJECT_DIR / "ویدیو"      # پوشه‌ی ویدیوهای ورودی
OUTPUT_DIR = PROJECT_DIR / "خروجی"     # خروجی‌ها اینجا ذخیره می‌شوند
MODELS_DIR = PROJECT_DIR / "models"
OUTPUT_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

# مدل خودرو: yolo11s — recall خیلی بهتر از nano روی صحنه‌ی شلوغ
# اگر فایل کنار پروژه بود از همان، وگرنه ultralytics خودش دانلود می‌کند (~۱۸MB)
_v11s = PROJECT_DIR / "yolo11s.pt"
VEHICLE_MODEL_PATH = str(_v11s) if _v11s.exists() else "yolo11s.pt"

PLATE_MODEL_PATH = MODELS_DIR / "license_plate_model.pt"
PLATE_MODEL_URL = (
    "https://github.com/Muhammad-Zeerak-Khan/"
    "Automatic-License-Plate-Recognition-using-YOLOv8/raw/main/license_plate_detector.pt"
)

# ============================================================================
#  حالت کشور  —  COUNTRY_MODE  (یکی از: "IR" | "GLOBAL" | "AUTO")
# ============================================================================
#   "IR"     → تحلیل کامل پلاک ایران: ارقام/حرف، کد استان، رنگ، نوع پلاک، منطقه آزاد
#   "GLOBAL" → فقط تشخیص پلاک + خواندن متن لاتین، بدون تحلیل کشوری
#   "AUTO"   → برای هر پلاک: پلاک ایرانیِ معتبر → تحلیل ایران، وگرنه → لاتین
# ============================================================================
SUPPORTED_COUNTRIES = {
    "IR": "ایران 🇮🇷 (تحلیل کامل در این نوت‌بوک)",
    "AE": "امارات 🇦🇪", "BR": "برزیل 🇧🇷", "DE": "آلمان 🇩🇪", "ES": "اسپانیا 🇪🇸",
    "FR": "فرانسه 🇫🇷", "GB": "بریتانیا 🇬🇧", "IN": "هند 🇮🇳", "IT": "ایتالیا 🇮🇹",
    "NL": "هلند 🇳🇱", "TR": "ترکیه 🇹🇷", "US": "آمریکا 🇺🇸",
}

COUNTRY_MODE = "AUTO"

# --- وسایل نقلیه (کلاس‌های COCO) ---
VEHICLE_NAMES = {
    1: "Bicycle", 2: "Car", 3: "Motorcycle", 5: "Bus", 6: "Train", 7: "Truck",
}
VEHICLE_CLASSES = list(VEHICLE_NAMES)

# --- پارامترهای تشخیص وسیله‌ی نقلیه ---
VEHICLE_CONF = 0.30              # آستانه‌ی پایین‌تر → خودروهای بیشتری (نسخه‌ی ۱: 0.5)
VEHICLE_IMGSZ = 1280            # وضوح بالاتر → خودروهای دور هم دیده می‌شوند
TRACKER_CFG = "bytetrack.yaml"  # ردیاب سبک داخلی ultralytics؛ به هر خودرو یک ID می‌دهد

# --- پارامترهای پلاک / OCR ---
# ترتیب اولویت: ۱) همیشه خودرو تشخیص داده شود  ۲) همیشه جعبه‌ی پلاک نشان داده شود (اگر پیدا شد)
# ۳) متن فقط وقتی نشان داده می‌شود که واقعاً خوانده شده باشد — وگرنه فقط تشخیص (بدون متن) کافی است.
PLATE_CONF = 0.40
PLATE_EVERY = 2     # هر چند فریم یک‌بار دنبال پلاک بگرد (۱ = همه‌ی فریم‌ها). سقفی برای «تلاش OCR» وجود ندارد —
                    # تا وقتی خودرو دیده می‌شود، اگر هنوز متن خوبی نخوانده، هر PLATE_EVERY فریم دوباره امتحان می‌شود.
FRAME_STRIDE = 1    # >1 یعنی هر N فریم پردازش شود (سریع‌تر؛ جعبه‌ها کمی عقب می‌افتند)

PROGRESS_EVERY = 30

IR_LOGIC = COUNTRY_MODE in ("IR", "AUTO")
OCR_LANGS = ["fa", "en"] if IR_LOGIC else ["en"]

print(f"✅ حالت: {COUNTRY_MODE}  |  مدل خودرو: {Path(VEHICLE_MODEL_PATH).name} @ {VEHICLE_IMGSZ}"
      f"  |  conf={VEHICLE_CONF}  |  tracker={TRACKER_CFG}")
print(f"   وسایل نقلیه: {', '.join(VEHICLE_NAMES.values())}")

## بخش ۳ — انتخاب خودکار GPU/CPU و بارگذاری مدل‌ها

In [ ]:
# انتخاب خودکار: اگر کارت گرافیک CUDA موجود باشد → GPU، وگرنه → CPU
USE_GPU = torch.cuda.is_available()
DEVICE = 0 if USE_GPU else "cpu"
print(f"🖥️ دستگاه: {'GPU (' + torch.cuda.get_device_name(0) + ')' if USE_GPU else 'CPU'}")

In [ ]:
# دانلود مدل پلاک (فقط بار اول)
if not PLATE_MODEL_PATH.exists():
    print("📥 در حال دریافت مدل پلاک از گیت‌هاب...")
    req = urllib.request.Request(PLATE_MODEL_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(PLATE_MODEL_PATH, "wb") as f:
        f.write(resp.read())
    print(f"✅ ذخیره شد: {PLATE_MODEL_PATH}")
else:
    print(f"✅ مدل پلاک موجود است: {PLATE_MODEL_PATH}")

In [ ]:
vehicle_model = YOLO(VEHICLE_MODEL_PATH)
plate_model = YOLO(str(PLATE_MODEL_PATH))

# --- OCR فارسی: Hezar CRNN (اگر لود نشد، EasyOCR) ---
easy_reader = easyocr.Reader(OCR_LANGS, gpu=USE_GPU)
hezar_model = None
if IR_LOGIC:
    try:
        from hezar.models import Model
        hezar_model = Model.load("hezarai/crnn-fa-license-plate-recognition-v2")
        print("✅ OCR فارسی: Hezar CRNN")
    except Exception as e:
        print("=" * 70)
        print(f"⚠️⚠️⚠️  Hezar لود نشد ({e})")
        print("    fallback به EasyOCR فعال شد — دقتِ خواندن پلاک فارسی به‌شدت پایین‌تر می‌آید.")
        print("    معمولاً یعنی اتصال به HuggingFace آن لحظه قطع بوده؛ این سلول را دوباره اجرا کن.")
        print("=" * 70)

# --- OCR لاتین: fast-plate-ocr (مدل جهانی، بسیار دقیق برای پلاک انگلیسی) ---
latin_ocr = None
if COUNTRY_MODE in ("GLOBAL", "AUTO"):
    try:
        from fast_plate_ocr import LicensePlateRecognizer
        # cct-s = دقیق‌تر و کمی سنگین‌تر؛ cct-xs = سبک‌تر
        latin_ocr = LicensePlateRecognizer("cct-s-v2-global-model")
        print("✅ OCR لاتین: fast-plate-ocr (cct-s-v2-global)")
    except Exception as e:
        print(f"⚠️ fast-plate-ocr لود نشد ({e}) — fallback: EasyOCR")

print(f"✅ مدل خودرو: {Path(VEHICLE_MODEL_PATH).name}  |  حالت: {COUNTRY_MODE}")

## بخش ۴ — توابع کمکی

In [ ]:
from PIL import Image, ImageDraw, ImageFont

try:
    import arabic_reshaper
    from bidi.algorithm import get_display
    _HAS_RTL = True
except Exception:
    _HAS_RTL = False

# فونت کوچک‌تر از نسخه‌ی ۱ چون صحنه شلوغ است و برچسب‌ها روی هم می‌افتند
try:
    _FONT = ImageFont.truetype(r"C:\Windows\Fonts\tahoma.ttf", 15)
except Exception:
    _FONT = ImageFont.load_default()

_DIGITS_TO_ASCII = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
_is_fa = lambda s: any("؀" <= c <= "ۿ" for c in s)


def _shape_fa(text):
    """آماده‌سازی متن فارسی برای رندر صحیح (اتصال حروف + راست‌به‌چپ)."""
    if _HAS_RTL and _is_fa(text):
        try:
            return get_display(arabic_reshaper.reshape(text))
        except Exception:
            return text
    return text


def draw_box(frame, x1, y1, x2, y2, color):
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)


def draw_plate_label(frame, x1, y1, text, color_bgr):
    """برچسب بالای جعبه را با Pillow می‌کشد تا فارسی/ارقام فارسی درست نشان داده شوند."""
    if not text:
        return
    disp = _shape_fa(str(text))
    img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    d = ImageDraw.Draw(img)
    l, t, r, b = d.textbbox((0, 0), disp, font=_FONT)
    tw, th = r - l, b - t
    ty = max(0, y1 - th - 8)
    fill = (color_bgr[2], color_bgr[1], color_bgr[0])
    d.rectangle([x1, ty, x1 + tw + 8, ty + th + 8], fill=fill)
    d.text((x1 + 4, ty + 3), disp, font=_FONT, fill=(0, 0, 0))
    frame[:] = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


def _upscale(crop, min_h=64):
    """پلاک‌های کوچک را بزرگ می‌کند — دقت CRNN/OCR را خیلی بالا می‌برد."""
    h, w = crop.shape[:2]
    if 0 < h < min_h:
        f = min_h / h
        crop = cv2.resize(crop, (int(w * f), min_h), interpolation=cv2.INTER_CUBIC)
    return crop


# ---------- موتورهای OCR ----------
def _ocr_easy(crop):
    res = easy_reader.readtext(_upscale(crop))
    if not res:
        return "", 0.0
    res.sort(key=lambda r: r[0][0][0])
    return " ".join(r[1] for r in res).strip(), float(np.mean([r[2] for r in res]))


def _ocr_hezar(crop):
    out = hezar_model.predict(_upscale(crop))
    while isinstance(out, (list, tuple)) and out:
        out = out[0]
    if isinstance(out, dict):
        return str(out.get("text", "")).strip(), float(out.get("score", 0.9) or 0.9)
    if hasattr(out, "text"):
        return str(out.text).strip(), float(getattr(out, "score", 0.9) or 0.9)
    return (str(out).strip() if out is not None else ""), 0.9


def _ocr_fastplate(crop):
    crop = _upscale(crop)
    if crop.ndim == 2:
        crop = cv2.cvtColor(crop, cv2.COLOR_GRAY2BGR)
    try:
        out = latin_ocr.run(crop, return_confidence=True)
    except TypeError:
        out = latin_ocr.run(crop)
    if isinstance(out, tuple) and len(out) >= 2:
        out = out[0]
    item = out[0] if isinstance(out, (list, tuple)) and out else out
    text = str(getattr(item, "plate", item if isinstance(item, str) else "") or "").replace("_", "").strip()
    probs = getattr(item, "char_probs", None)
    arr = np.asarray(probs, float).reshape(-1) if probs is not None else np.array([])
    conf = float(arr.mean()) if arr.size else 0.9
    return text, max(0.0, min(1.0, conf))


def _read_fa(crop):
    return _ocr_hezar(crop) if hezar_model is not None else _ocr_easy(crop)


def _read_latin(crop):
    return _ocr_fastplate(crop) if latin_ocr is not None else _ocr_easy(crop)


def read_plate(crop):
    """تصویر پلاک → (متن، اطمینان، script).  script: 'fa' یا 'latin'."""
    if crop is None or crop.size == 0:
        return "", 0.0, "latin"
    try:
        if COUNTRY_MODE == "IR":
            t, c = _read_fa(crop)
            return t, c, "fa"
        if COUNTRY_MODE == "GLOBAL":
            t, c = _read_latin(crop)
            return t, c, "latin"
        # AUTO: اول فارسی؛ اگر پلاک ایرانیِ معتبر بود همان، وگرنه موتور دقیق لاتین
        t_fa, c_fa = _read_fa(crop)
        if parse_iranian_plate(t_fa, crop)["valid"]:
            return t_fa, c_fa, "fa"
        t_lat, c_lat = _read_latin(crop)
        if t_lat:
            return t_lat, c_lat, "latin"
        return t_fa, c_fa, "fa"
    except Exception:
        t, c = _ocr_easy(crop)
        return t, c, ("fa" if _is_fa(t) else "latin")

## بخش ۵ — داده‌ها و منطق پلاک ایران

منبع: `PelakX/configs/countries/ir.yaml` (کد استان‌ها از ghabzino + ویکی‌پدیا، معناشناسی حرف از نمونه‌های عکاسی‌شده).
در حالت `IR` روی همه‌ی پلاک‌ها اجرا می‌شود؛ در حالت `AUTO` فقط روی پلاک‌هایی که خط فارسی دارند.

In [ ]:
# کد دورقمی استان (کدهای مشترک عیناً مطابق سیستم واقعی)
PROVINCE_CODES = {
    "10": "تهران", "11": "تهران", "12": "خراسان رضوی", "13": "اصفهان", "14": "خوزستان",
    "15": "آذربایجان شرقی", "16": "قم", "17": "آذربایجان غربی", "18": "همدان", "19": "کرمانشاه",
    "20": "تهران", "21": "البرز", "22": "تهران", "23": "اصفهان", "24": "خوزستان",
    "25": "آذربایجان شرقی", "26": "خراسان شمالی", "27": "آذربایجان غربی", "28": "همدان",
    "29": "کرمانشاه", "30": "تهران/البرز", "31": "لرستان", "32": "خراسان رضوی/شمالی/جنوبی",
    "33": "تهران", "34": "خوزستان", "35": "آذربایجان شرقی", "36": "خراسان رضوی",
    "37": "آذربایجان غربی", "38": "البرز", "39": "کرمانشاه", "40": "تهران", "41": "لرستان",
    "42": "خراسان رضوی", "43": "اصفهان", "44": "تهران", "45": "کرمان", "46": "گیلان",
    "47": "مرکزی", "48": "بوشهر", "49": "کهگیلویه و بویراحمد", "50": "تهران", "51": "کردستان",
    "52": "خراسان جنوبی", "53": "اصفهان", "54": "یزد", "55": "تهران", "56": "گیلان",
    "57": "مرکزی", "58": "بوشهر", "59": "گلستان", "60": "تهران", "61": "کردستان",
    "62": "مازندران", "63": "فارس", "64": "یزد", "65": "کرمان", "66": "تهران",
    "67": "اصفهان", "68": "البرز", "69": "گلستان", "71": "چهارمحال و بختیاری",
    "72": "مازندران", "73": "فارس", "74": "خراسان رضوی/شمالی", "75": "کرمان",
    "76": "گیلان", "77": "تهران", "78": "تهران/البرز", "79": "قزوین",
    "81": "چهارمحال و بختیاری", "82": "مازندران", "83": "فارس", "84": "هرمزگان",
    "85": "سیستان و بلوچستان", "86": "سمنان", "87": "زنجان", "88": "تهران", "89": "قزوین",
    "91": "اردبیل", "92": "مازندران", "93": "فارس", "94": "هرمزگان", "95": "سیستان و بلوچستان",
    "96": "سمنان", "97": "زنجان", "98": "ایلام", "99": "تهران",
}

# معنای حرف پلاک (نوع) + رنگ مورد انتظار زمینه
LETTER_SEMANTICS = {
    "الف": ("دولتی", "قرمز"), "پ": ("پلیس/انتظامی", "سبز"),
    "ت": ("تاکسی", "زرد"), "ع": ("حمل‌ونقل عمومی", "زرد"),
    "ک": ("ماشین‌آلات کشاورزی", "زرد"), "ژ": ("جانبازان و معلولین", "سفید"),
    "گ": ("گذر موقت", "سفید"), "ث": ("سپاه (تأیید‌نشده)", "?"),
    "D": ("دیپلمات", "آبی"), "S": ("سیاسی", "آبی"),
    "معلولین": ("جانبازان و معلولین", "سفید"),
    "تشریفات": ("تشریفات", "قرمز"), "موقت": ("موقت منطقه آزاد", "?"),
}

# حروف مجاز در جایگاه حرف پلاک شخصی/عمومی
IR_LETTERS = list("بپتثجحدزژسشصطعفقکگلمنوهی") + ["الف", "D", "S"]

FA_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
NOISE_WORDS = ["ایران", "IRAN", "IR", "منطقه آزاد", "آزاد"]
print("✅ داده‌های پلاک ایران بارگذاری شد.")

In [ ]:
def detect_plate_color(plate_crop):
    """رنگ غالب زمینه‌ی پلاک را برمی‌گرداند: سفید/زرد/قرمز/سبز/آبی/نامشخص."""
    if plate_crop is None or plate_crop.size == 0:
        return "نامشخص"
    hsv = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2HSV)
    h, s, v = (int(np.median(hsv[:, :, i])) for i in range(3))
    if s < 60 and v > 120:
        return "سفید"
    if s < 60:
        return "نامشخص"
    if h < 12 or h > 168:
        return "قرمز"
    if 15 <= h <= 38:
        return "زرد"
    if 40 <= h <= 85:
        return "سبز"
    if 90 <= h <= 140:
        return "آبی"
    return "نامشخص"


def parse_iranian_plate(raw_text, plate_crop):
    """متن خام OCR + تصویر پلاک → دیکشنری تحلیل‌شده."""
    t = raw_text.translate(_DIGITS_TO_ASCII)
    for w in NOISE_WORDS:
        t = t.replace(w, " ")
    digits = "".join(c for c in t if c.isdigit())
    letters = (["الف"] if "الف" in t else []) + [c for c in t if c in IR_LETTERS]
    letter = letters[0] if letters else ""

    color = detect_plate_color(plate_crop)
    plate_type, _ = LETTER_SEMANTICS.get(letter, ("شخصی", "سفید"))
    if not letter and color == "زرد":
        plate_type = "عمومی/تاکسی (از روی رنگ)"

    out = {
        "raw": raw_text, "digits": digits, "letter": letter,
        "color": color, "type": plate_type, "province": "", "province_code": "",
        "free_zone": ("منطقه آزاد" in raw_text or "موقت" in raw_text),
        "formatted": raw_text, "valid": False,
    }

    if len(digits) >= 7 and letter:          # چیدمان شخصی/عمومی: DD L DDD + کد استان DD
        left, right, prov = digits[:2], digits[2:5], digits[5:7]
        out["province_code"] = prov
        out["province"] = PROVINCE_CODES.get(prov, "نامشخص")
        out["formatted"] = f"{left} {letter} {right} - ایران {prov}"
        out["valid"] = True
    elif len(digits) == 8 and not letter:    # موتورسیکلت
        out["type"] = "موتورسیکلت"
        out["province_code"] = digits[:3]
        out["formatted"] = f"{digits[:3]} - {digits[3:]}"
        out["valid"] = True

    return out


def ir_video_label(info):
    """برچسب روی ویدیو برای پلاک ایرانی (با Pillow رندر می‌شود، پس فارسی مشکلی ندارد)."""
    if info["valid"]:
        return info["formatted"]
    return info["digits"] or info["raw"] or "پلاک"


def latin_video_label(text):
    return text or "PLATE"

## بخش ۶ — تابع اصلی پردازش ویدیو (ByteTrack)

اولویت‌ها (به همین ترتیب، هرکدام مستقل از بعدی است):
1. **خودرو همیشه تشخیص داده می‌شود** — با `vehicle_model.track(...)`؛ هر خودرو یک **ID پایدار**.
2. **جعبه‌ی پلاک همیشه نشان داده می‌شود** وقتی پیدا شود — چه متنش خوانده شده باشد چه نه.
3. **متن فقط وقتی نشان داده می‌شود که واقعاً درست خوانده شده باشد**؛ در غیر این صورت فقط تشخیص (کادر) کافی است، چیزی حدس زده نمی‌شود.

نکته‌ی مهم نسبت به نسخه‌ی اول تلاشِ این بخش: **سقف دائمی برای OCR وجود ندارد.** تا وقتی خودرو دیده می‌شود
و متنِ خوبی هنوز قفل نشده، هر `PLATE_EVERY` فریم دوباره امتحان می‌شود — پس اگر پلاک در فریم‌های اول دور/مایل
بود ولی بعداً واضح شد، باز هم خوانده می‌شود (باگ نسخه‌ی قبلی همین بود: بعد از چند تلاشِ ناموفق برای همیشه
از آن خودرو صرف‌نظر می‌کرد).

In [ ]:
CSV_HEADER = ["id", "vehicle", "veh_conf", "plate_found", "plate_read", "script",
              "raw", "digits", "letter", "color", "type", "province", "formatted",
              "ocr_conf", "frames_seen"]

MIN_FRAMES = 3       # خودرویی که کمتر از این تعداد فریم دیده شده، در CSV نمی‌آید
MIN_LABEL_W = 45     # جعبه‌ی باریک‌تر از این فقط کادر می‌گیرد، بدون متن (کاهش شلوغی)


def _good_enough(text, conf, script):
    """آیا این خواندن آن‌قدر خوب هست که «قفل» شود؟ (فقط برای نمایش/توقفِ OCR — تشخیص پلاک همیشه ادامه دارد)"""
    if not text:
        return False
    if script == "fa":
        return parse_iranian_plate(text, None)["valid"] and conf >= 0.50
    return conf >= 0.90 and len(text) >= 5


def process_video(video_path, output_path, csv_path=None, max_frames=None, frame_stride=None):
    """ویدیو → ویدیوی حاشیه‌دار + CSV (یک ردیف برای هر خودروی دیده‌شده).

    اولویت: ۱) خودرو همیشه تشخیص  ۲) جعبه‌ی پلاک همیشه نشان داده می‌شود اگر پیدا شود
    ۳) متن فقط وقتی نشان داده می‌شود که واقعاً خوانده شده باشد.
    max_frames / frame_stride: فقط برای تست سریع.
    """
    stride = frame_stride or FRAME_STRIDE
    video_path, output_path = str(video_path), str(output_path)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"ویدیو باز نشد: {video_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

    veh = {}              # track_id -> state
    last_annotated = None
    ocr_calls = 0
    print(f"🚀 پردازش {Path(video_path).name} — {total} فریم | "
          f"{Path(VEHICLE_MODEL_PATH).name} @ {VEHICLE_IMGSZ} | حالت {COUNTRY_MODE}")

    frame_i = 0
    while True:
        ret, frame = cap.read()
        if not ret or (max_frames and frame_i >= max_frames):
            break
        frame_i += 1

        if stride > 1 and (frame_i % stride) != 0:
            out.write(last_annotated if last_annotated is not None else frame)
            continue
        if frame_i % PROGRESS_EVERY == 0:
            print(f"⏳ فریم {frame_i}/{total}  (ترک‌ها: {len(veh)} | OCR: {ocr_calls})")

        # --- ۱) خودرو: همیشه تشخیص + ردیابی ---
        res = vehicle_model.track(
            frame, persist=(frame_i > 1), tracker=TRACKER_CFG, classes=VEHICLE_CLASSES,
            conf=VEHICLE_CONF, imgsz=VEHICLE_IMGSZ, verbose=False, device=DEVICE,
        )[0]

        boxes = res.boxes
        ids = boxes.id.int().tolist() if boxes.id is not None else []
        for b, tid in zip(boxes, ids):
            cls, conf = int(b.cls[0]), float(b.conf[0])
            x1, y1, x2, y2 = map(int, b.xyxy[0])

            st = veh.get(tid)
            if st is None:
                st = {"votes": {}, "peak": {}, "frames": 0, "vehicle": "Vehicle",
                      "veh_conf": 0.0, "plate_found": False, "text": "", "conf": -1.0,
                      "script": "", "info": None, "color": "", "locked": False}
                veh[tid] = st
            st["frames"] += 1
            st["votes"][cls] = st["votes"].get(cls, 0.0) + conf
            st["peak"][cls] = max(st["peak"].get(cls, 0.0), conf)
            scls = max(st["votes"], key=st["votes"].get)
            st["vehicle"] = VEHICLE_NAMES.get(scls, "Vehicle")
            st["veh_conf"] = st["peak"][scls]

            draw_box(frame, x1, y1, x2, y2, (255, 150, 0))

            # --- ۲) پلاک: همیشه دنبالش بگرد (تا وقتی متن خوب قفل نشده)؛ بدون سقف تلاش ---
            plate_box = None
            if scls not in (1, 6) and (frame_i % PLATE_EVERY) == 0:
                vc = frame[max(0, y1):y2, max(0, x1):x2]
                if vc.size:
                    pr = plate_model.predict(vc, verbose=False, conf=PLATE_CONF, device=DEVICE)[0]
                    if len(pr.boxes):
                        st["plate_found"] = True
                        pb = max(pr.boxes, key=lambda p: float(p.conf[0]))
                        px1, py1, px2, py2 = map(int, pb.xyxy[0])
                        plate_box = (x1 + px1, y1 + py1, x1 + px2, y1 + py2)  # جعبه همیشه ثبت می‌شود

                        # --- ۳) متن: فقط اگر هنوز قفل نشده، OCR را امتحان کن ---
                        if not st["locked"]:
                            pimg = vc[py1:py2, px1:px2]
                            t, c, sc = read_plate(pimg)
                            ocr_calls += 1
                            if t and c > st["conf"]:
                                st["text"], st["conf"], st["script"] = t, c, sc
                                st["color"] = detect_plate_color(pimg)
                                st["info"] = parse_iranian_plate(t, pimg) if sc == "fa" else None
                            if _good_enough(st["text"], st["conf"], st["script"]):
                                st["locked"] = True

            # --- برچسب: جعبه‌ی خودرو + ID همیشه؛ متن فقط وقتی واقعاً خوانده شده ---
            if (x2 - x1) >= MIN_LABEL_W:
                lbl = f"#{tid} {st['vehicle']}"
                if _good_enough(st["text"], st["conf"], st["script"]):
                    pl = (ir_video_label(st["info"]) if st["script"] == "fa" and st["info"]
                          else latin_video_label(st["text"]))
                    lbl += f" | {pl}"
                draw_plate_label(frame, x1, y1, lbl, (255, 150, 0))
            if plate_box:
                draw_box(frame, *plate_box, (0, 255, 0))   # کادر پلاک همیشه، حتی بدون متن خوب

        out.write(frame)
        last_annotated = frame

    cap.release()
    out.release()

    # --- CSV: یک ردیف برای هر خودروی دیده‌شده (نه فقط آن‌ها که پلاک خوانده شد) ---
    rows = []
    for tid, st in veh.items():
        if st["frames"] < MIN_FRAMES:
            continue
        good = _good_enough(st["text"], st["conf"], st["script"])
        info = st["info"] if good else None
        head = [tid, st["vehicle"], round(st["veh_conf"], 3), st["plate_found"], good]
        if good and st["script"] == "fa" and info:
            rows.append(head + ["fa", info["raw"], info["digits"], info["letter"],
                                info["color"], info["type"], info["province"],
                                info["formatted"], round(st["conf"], 3), st["frames"]])
        elif good:
            rows.append(head + [st["script"], st["text"], "", "", st["color"], "", "",
                                st["text"], round(st["conf"], 3), st["frames"]])
        else:   # خودرو (و شاید جعبه‌ی پلاک) دیده شد، ولی متنِ قابل‌اعتماد خوانده نشد
            rows.append(head + ["", "", "", "", "", "", "", "", "", st["frames"]])

    if csv_path:
        with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.writer(f)
            w.writerow(CSV_HEADER)
            w.writerows(rows)
        n_read = sum(1 for r in rows if r[4])
        print(f"📄 CSV: {Path(csv_path).name}  ({len(rows)} خودرو، {n_read} پلاکِ خوانده‌شده)")

    print(f"✅ خروجی: {output_path}   |   فراخوانی OCR: {ocr_calls}")
    return output_path, rows

## بخش ۷ — تست روی ویدیوها

آدرس ویدیوهای تست را در `TEST_VIDEOS` بگذارید (مسیر کامل یا نام فایل داخل پوشه‌ی `ویدیو`).
برای هر ویدیو یک `<name>_out.mp4` و یک `<name>_plates.csv` در پوشه‌ی `خروجی` ساخته می‌شود.

In [ ]:
TEST_VIDEOS = [
    VIDEO_DIR / "1_P.mp4",
    VIDEO_DIR / "2-1_P.mp4",
    VIDEO_DIR / "2-2_P.mp4",
    # مسیر ویدیوهای جدید را اینجا اضافه کنید
]

results = []
for video in TEST_VIDEOS:
    video = Path(video)
    if not video.exists():
        print(f"⚠️ پیدا نشد: {video}")
        continue
    results.append(process_video(
        video,
        OUTPUT_DIR / f"{video.stem}_v2_out.mp4",
        OUTPUT_DIR / f"{video.stem}_v2_plates.csv",
        max_frames=360,     # برای تست سریع فعال کن
    ))

In [ ]:
# خلاصه‌ی خودروهای دیده‌شده در هر ویدیو (چه پلاکشان خوانده شده باشد چه نه)
import pandas as pd

for out_path, rows in results:
    n_read = sum(1 for r in rows if r[4])
    print(f"\n🎥 {Path(out_path).name}  —  {len(rows)} خودرو  |  {n_read} پلاکِ خوانده‌شده")
    if not rows:
        continue
    df = pd.DataFrame(rows, columns=CSV_HEADER).sort_values("frames_seen", ascending=False)
    display(df[["id", "vehicle", "veh_conf", "plate_found", "plate_read",
                "formatted", "province", "color", "ocr_conf", "frames_seen"]])

In [ ]:
# نمایش ویدیوی خروجی درون نوت‌بوک
# (اگر مرورگر کدک mp4v را پخش نکرد، فایل را مستقیم از پوشه‌ی خروجی باز کنید)
from IPython.display import Video, display

for out_path, _ in results:
    display(Video(str(out_path), embed=True, width=640))

## بخش ۸ — ساخت GIF از خروجی

از هر ویدیوی خروجی یک GIF کوتاه می‌سازد (برای اشتراک‌گذاری). در پوشه‌ی `خروجی` ذخیره می‌شود.

In [ ]:
import imageio.v2 as imageio
from IPython.display import Image as IPyImage


def video_to_gif(mp4_path, gif_path, out_fps=8, scale=0.5, max_seconds=6):
    """ویدیو → GIF کوتاه و سبک."""
    cap = cv2.VideoCapture(str(mp4_path))
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 25
    step = max(1, round(src_fps / out_fps))
    max_frames = int(src_fps * max_seconds)

    frames, i = [], 0
    while True:
        ret, frame = cap.read()
        if not ret or i > max_frames:
            break
        if i % step == 0:
            if scale != 1:
                frame = cv2.resize(frame, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        i += 1
    cap.release()

    imageio.mimsave(str(gif_path), frames, fps=out_fps, loop=0)
    mb = Path(gif_path).stat().st_size / 1e6
    print(f"🎞️ {Path(gif_path).name}  —  {len(frames)} فریم، {mb:.1f} MB")
    return gif_path


for out_path, _ in results:
    gif = Path(out_path).with_suffix("").with_name(Path(out_path).stem + ".gif")
    video_to_gif(out_path, gif)
    display(IPyImage(filename=str(gif)))